# 1. Conjunto de dados
Para esse exerício, baixe o conjunto de dados BBC News Classification diposnível em https://www.kaggle.com/competitions/learn-ai-bbc/data

1.1. Baixe o dataset, carregue as amostras de treino e teste em um dataframe.

### 1.1.1 Configurar o Token da API do Kaggle

Para interagir com a API do Kaggle e baixar datasets, você precisa configurar sua chave de API. Por segurança, é recomendado usar o gerenciador de segredos do Google Colab.

1.  Clique no ícone **🔑 Secrets** no painel esquerdo.
2.  Clique em **+ New secret**.
3.  No campo **Name**, digite `KAGGLE_API_TOKEN`.
4.  No campo **Value**, cole o valor do seu token da API do Kaggle (ex: `KGAT_ce5c3e3c2f9c7c91f7e88b5b555d4971`).
5.  Certifique-se de que a opção **Notebook access** esteja ativada.

In [ ]:
import os
from google.colab import userdata

# Recupera o token da API do Kaggle do gerenciador de segredos
KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')

# Define o token como uma variável de ambiente para que kagglehub possa encontrá-lo
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

print("Kaggle API token configurado com sucesso!")

In [ ]:
import kagglehub

# Download latest version
# If you encounter a 401 Unauthorized error, please ensure you have accepted the competition rules on the Kaggle website:
# https://www.kaggle.com/competitions/learn-ai-bbc/rules
path = kagglehub.competition_download('learn-ai-bbc')

print("Path to competition files:", path)

In [ ]:
import os
import pandas as pd

# List the contents of the downloaded directory
print(f"Contents of {path}: {os.listdir(path)}")

try:
    # Load both training and testing datasets
    train_df = pd.read_csv(os.path.join(path, 'BBC News Train.csv'))

    print("Train data loaded successfully:")
    print(train_df.head())

except FileNotFoundError:
    print(f"Error: Data files not found in {path}. Please check the filenames.")
    print("Expected files like 'BBC News Train.csv' and 'BBC News Test.csv'")

### 1.2. Extrair Textos e Categorias do conjunto de dados da BBC News

In [ ]:
import numpy as np

# Extract texts and labels from train_df
bbc_train_texts = train_df['Text'].tolist()
bbc_train_labels_str = train_df['Category'].tolist()


print(f"Primeiros 5 textos de treino da BBC: {bbc_train_texts[:1]}")
print(f"Primeiras 5 labels de treino da BBC: {bbc_train_labels_str[:5]}")


###Q1) Explore os dados.
* a. Quantos textos existem no dataset?
* b. Desenhe um diagrama de barras com a quantidade de textos em cada classe. Responda.
* c. As classes possuem quantidades semelhantes de exemplos?
* d. Qual categoria possui mais registros?
* e. Qual categoria possui menos registros?
* f. Existe desbalanceamento entre classes?
* g. Exiba em um gráfico de caixa (boxplot) a distribuição do tamanho dos textos em cada classe. Responda:  Qual o tamanho médio dos textos?



In [ ]:
#codifique aqui

### Separe 90% dos dados para treino e 10% para teste. Armazene o texto em bbc_train_texts e bbc_test_texts e as labels em y_bbc_train e y_bbc_test

In [ ]:
#codifique aqui

### Convertendo labels categóricas em numéricas

Vamos mapear as categorias para inteiros para que possam ser usadas no treinamento do modelo.

In [ ]:
import numpy as np

# Extract texts and labels from train_df (already done by d6917e78, assuming data_df is train_df for simplicity here)
# bbc_train_texts and bbc_train_labels_str are available from cell d6917e78
bbc_train_texts = train_df['Text'].tolist()
bbc_train_labels_str = train_df['Category'].tolist()

# Faça o mesmo para os dados de teste criando bbc_test_labels_str, alterando a linha a seguir
bbc_test_labels_str=[]

# identificar categorias únicas
all_categories = sorted(list(set(bbc_train_labels_str + bbc_test_labels_str)))
category_to_int = {category: i for i, category in enumerate(all_categories)}


# Converter labels para números
bbc_train_labels = np.array([category_to_int[cat] for cat in bbc_train_labels_str])

# Converta labels de teste


### Camada de vetorização para os dados da BBC News

Observe o código abaixo e responda às perguntas.

* Q2) Como vocabulário pequeno demais pode prejudicar resultados?
* Q3) Como sequências muito curtas podem afetar classificação?
* Q4) Sequências muito longas trazem quais custos?
* Q5) Altere o código abaixo da camada de vetorização para considerar um  tamanho de vocabulário, bem como tamanho de textos mais adequados ao conjunto de dados em questão.


In [ ]:
from tensorflow.keras.layers import TextVectorization

# Camada de vetorização específica para o conjunto de dados da BBC News.
bbc_vectorize_layer = TextVectorization(
    max_tokens=100,
    output_mode='int',
    output_sequence_length=64,
    standardize='lower_and_strip_punctuation',
    split='whitespace'
)

# Adaptar a camada de vetorização aos textos de treino da BBC News
bbc_vectorize_layer.adapt(np.array(bbc_train_texts))

# Transformar os textos de treino da BBC News em sequências de inteiros preenchidas
X_bbc_train = bbc_vectorize_layer(np.array(bbc_train_texts)).numpy()

# Atribuir as labels extraídas da BBC News para y_bbc_train
y_bbc_train = bbc_train_labels

print(f"Shape de X_bbc_train (textos processados da BBC News): {X_bbc_train.shape}")
print(f"Shape de y_bbc_train (labels da BBC News): {y_bbc_train.shape}")

# Also print info for labels after conversion
print(f"Mapeamento de categorias: {category_to_int}")
print(f"Primeiras 5 labels numéricas de treino da BBC: {bbc_train_labels[:5]}")

### Processar os dados de teste da BBC News

* Q6) Aplicar a mesma camada de vetorização aos textos de teste da BBC News.

In [ ]:
#codifique aqui

## Definindo a rede neural
A rede neural pode ser vista como uma sequencia de camadas de transformação, a saída de uma camada é dada como entrada para a próxima camada.

* Q7) Qual modificação foi necessária para adaptar a rede à classificação com muitas classes? Compare com o exemplo dado na aula anterior.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

# Get the number of unique categories from the category_to_int mapping
num_classes = len(category_to_int)

model = Sequential([
    Embedding(input_dim=10000, output_dim=16, input_length=256),
    Flatten(),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

* Q8) Modifique a arquitetura da rede incluindo outra camada densa intermediária e compile o modelo. Quantos parâmetros treináveis tem seu modelo?

* Q9) Observe a alteração feita na entrada do método compile em relação ao exemplo da aula. Qual alteração foi feita e por quê?

In [ ]:
import tensorflow as tf
import numpy as np

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## 4. Treinamento do modelo


* Q10) No método para o treinamento, altere o número de épocas para obter um melhor valor de loss.

In [ ]:

# Converte X e y para tensores no TensorFlow
X_tensor = tf.constant(X_bbc_train, dtype=tf.int32)
y_tensor = tf.constant(y_bbc_train, dtype=tf.int32) # Labels should be int for sparse_categorical_crossentropy

model.fit(X_tensor, y_tensor, epochs=10, verbose=1)

* Q11) Avalie o modelo usando a porção de dados de teste. Qual o resultado?



In [ ]:
# codifique aqui

* Q12) Mostre como predizer a classe de um texto específico.

In [ ]:
#codifique aqui